# Milestone-3

### Loading dataset

In [2]:
import pandas as pd
df=pd.read_csv(r"C:\Users\Deepalakshmi\Downloads\sample_emails_with_triage_200.csv")

### Displaying first 5 rows

In [3]:
df.head()

,id,sender,subject,body,priority,triage_label,ideal_intent,ideal_tone
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,notify,urgent
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,respond,polite
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,notify,urgent
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,respond,polite
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,respond,polite


### Using HITL Check

In [4]:
DANGEROUS_ACTIONS = ["respond"]

In [5]:
def hitl_check(action):
    if action in DANGEROUS_ACTIONS:
        return "WAIT_FOR_HUMAN" 
    return "AUTO_APPROVED"

In [6]:

#Simulate Human Approval
def human_decisions():
    decision = input("Approve action? (yes/no): ")
    return decision.lower() == "yes"    

### Email assistant logic

In [7]:
def email_assistant(email_text):
    text = str(email_text).lower()

    # URGENT / NOTIFY
    if any(word in text for word in [
        "urgent", "asap", "immediately", "submit", "deadline", "eod", "today"
    ]):
        return "notify", "urgent"

    # RESPOND / POLITE
    elif any(word in text for word in [
        "please", "can you", "could you", "kindly", "review", "confirm", "reply"
    ]):
        return "respond", "polite"

    # IGNORE / POLITE
    elif any(word in text for word in [
        "thank you", "thanks", "appreciate", "grateful"
    ]):
        return "ignore", "polite"

    # INFORMATIONAL / NEUTRAL
    elif any(word in text for word in [
        "meeting", "schedule", "reminder", "update", "announcement"
    ]):
        return "notify", "neutral"

    # DEFAULT (UNCLEAR)
    else:
        return "review", "neutral"


### Applying HITL Check to Entire Dataset

In [8]:
approved = input("Approve all WAIT_FOR_HUMAN actions? (yes/no): ").lower() == "yes"

results = []

for _, row in df.iterrows():
    action, tone = email_assistant(row["body"])
    status = hitl_check(action)

    if status == "WAIT_FOR_HUMAN":
        final_action = action if approved else "blocked"
    else:
        final_action = action

    results.append({
        "email": row["body"][:50],
        "ai_action": action,
        "final_action": final_action,
        "hitl_status": status,
        "tone": tone
    })

results_df = pd.DataFrame(results)
results_df


,email,ai_action,final_action,hitl_status,tone
0,Reminder: The client meeting is scheduled at 1...,notify,notify,AUTO_APPROVED,neutral
1,Your invoice of INR 25515.09 is due on 2025-12...,respond,respond,WAIT_FOR_HUMAN,polite
2,Reminder: The client meeting is scheduled at 1...,notify,notify,AUTO_APPROVED,neutral
3,"Hello team, please find the attached weekly re...",respond,respond,WAIT_FOR_HUMAN,polite
4,"Hello team, please find the attached weekly re...",respond,respond,WAIT_FOR_HUMAN,polite
...,...,...,...,...,...
195,Security alert: multiple failed login attempts...,review,review,AUTO_APPROVED,neutral
196,Your order #3828 has been shipped and is expec...,review,review,AUTO_APPROVED,neutral
197,Congratulations! You have been selected as a l...,review,review,AUTO_APPROVED,neutral
198,Please complete the mandatory training module ...,notify,notify,AUTO_APPROVED,urgent


### Saving output to csv file

In [14]:
output_path = r"C:\Users\Deepalakshmi\Downloads\milestone3_output_lingaeswari.csv"

results_df.to_csv(output_path, index=False)

print("Saved to:", output_path)

Saved to: C:\Users\Deepalakshmi\Downloads\milestone3_output_lingaeswari.csv
